# Python API Example - European Gas Contracts
## Calling the latest & historical Spark European Gas Prices

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint:
https://www.sparkcommodities.com/api/

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

__This section can remain unchanged for most Spark API users.__

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
from io import StringIO
import datetime

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. You can create and download API credentials from the Spark Platform:

https://app.sparkcommodities.com/data-integrations/api



In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)
print(access_token)

# 2. Fetching Contract Prices

Here we define the function used to call price releases for Spark Gas Contracts. This endpoint takes 5 parameters - details on these parameters can be found on the API docs website:

https://www.sparkcommodities.com/api/gas/contracts.html


Users can also choose which format to retrieve the data in, 'json' or 'csv'.

__N.B__ Metadata is only available via the JSON format.

In [ ]:
from io import StringIO

## Defining the function

def fetch_prices(access_token, ticker, unit, start, end, limit=None, offset=None, format='json'):

    query_params = '?unit={}'.format(unit)

    query_params += '&start={}'.format(start)

    query_params += '&end={}'.format(end)

    if limit is not None:
        query_params += '&limit={}'.format(limit)

    if offset is not None:
        query_params += '&offset={}'.format(offset)


    uri="v1.0/gas/contracts/{}/{}".format(ticker,query_params)
    print(uri)
    
    content = do_api_get_query(
        uri=uri, access_token=access_token, format=format
    )
    
    if format == 'json':
        my_dict = content
    elif format == 'csv':
        # if there's no data to show, returns raw response (empty string) and "No Data to Show" message
        if len(content) == 0:
            my_dict = content
            print('No Data to Show')
        else:
            my_dict = content.decode('utf-8')
            my_dict = pd.read_csv(StringIO(my_dict))  # automatically converting into a Pandas DataFrame when choosing CSV format
    
    return my_dict

In [ ]:
# calling the data as a JSON
data_json  = fetch_prices(access_token, ticker='sparkleba-ttf-fo', unit='eur-per-mwh', start='2026-01-01', end='2026-02-01', format='json')

# printing metaData
data_json['metaData']

In [ ]:
# calling data as a CSV
csv_df  = fetch_prices(access_token, ticker='sparkleba-ttf-fo', unit='eur-per-mwh', start='2026-01-01', end='2026-02-10', format='csv')
csv_df.head(5)

In [ ]:
# retrieving a single release date
df = csv_df[csv_df['ReleaseDate'] == '2026-01-30']
df.head(5)